<a href="https://colab.research.google.com/github/Noisy77-pixel/urdu-ocr-codesaviours-si26-bilal/blob/main/SI26_Week4_Bilal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!unzip data.zip

Archive:  data.zip
   creating: data/
  inflating: data/labels.csv         
   creating: data/processed/
   creating: data/raw/
   creating: data/raw/books/
   creating: data/raw/handwritten_chars/
  inflating: data/raw/handwritten_chars/hw_char_001.png  
  inflating: data/raw/handwritten_chars/hw_char_002.png  
  inflating: data/raw/handwritten_chars/hw_char_003.png  
  inflating: data/raw/handwritten_chars/hw_char_004.png  
  inflating: data/raw/handwritten_chars/hw_char_005.png  
  inflating: data/raw/handwritten_chars/hw_char_006.png  
  inflating: data/raw/handwritten_chars/hw_char_007.png  
  inflating: data/raw/handwritten_chars/hw_char_008.png  
  inflating: data/raw/handwritten_chars/hw_char_009.png  
  inflating: data/raw/handwritten_chars/hw_char_010.png  
  inflating: data/raw/handwritten_chars/hw_char_011.png  
  inflating: data/raw/handwritten_chars/hw_char_012.png  
  inflating: data/raw/handwritten_chars/hw_char_013.png  
  inflating: data/raw/handwritten_chars/hw_char_

In [2]:
!pip install sentencepiece transformers pillow pandas

In [1]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import ViTImageProcessor, RobertaTokenizer, TrOCRProcessor
from transformers import VisionEncoderDecoderModel
import os

print(f"PyTorch version: {torch.__version__}")

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

if device == 'cpu':
    print('⚠️ WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')
    print('Training on CPU will be extremely slow!')

print('Libraries loaded successfully!')

PyTorch version: 2.11.0+cu128
Using device: cuda
Libraries loaded successfully!


In [2]:
class UrduOCRDataset(Dataset):
    """
    PyTorch Dataset for Urdu OCR using TrOCR processor.
    Handles dynamic file path mapping to find images across subdirectories.
    """
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.path_mapping = {}

        print("Mapping image paths across all subdirectories...")
        for root, _, files in os.walk('data'):
            for f in files:
                if f.endswith(('.png', '.jpg', '.jpeg')):
                    self.path_mapping[f] = os.path.join(root, f)

        # Filter out missing files
        self.data['filename'] = self.data['image'].apply(os.path.basename)
        initial_count = len(self.data)
        self.data = self.data[self.data['filename'].isin(self.path_mapping.keys())].reset_index(drop=True)

        if len(self.data) < initial_count:
            print(f"⚠️ Filtered out {initial_count - len(self.data)} missing images.")
        print(f'✅ Final Dataset size: {len(self.data)} samples.')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        filename = row['filename']
        image_path = self.path_mapping[filename]

        image = Image.open(image_path).convert('RGB')

        # Use return_tensors='pt', fallback to np if torch link fails
        try:
            encoding = self.processor(image, return_tensors='pt')
            pixel_values = encoding.pixel_values.squeeze()
        except Exception:
            encoding = self.processor(image, return_tensors='np')
            pixel_values = torch.from_numpy(encoding.pixel_values).squeeze()

        # Tokenize the text label
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128,
            truncation=True
        ).input_ids

        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}

print('UrduOCRDataset class defined successfully!')

UrduOCRDataset class defined successfully!


In [5]:
image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-printed')
tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed', use_fast=False)
processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
print('✅ TrOCR processor loaded!')

# Load the pretrained model
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

# Configure model for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print(f'✅ Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Create dataset and train/test split
dataset = UrduOCRDataset('data/labels.csv', processor)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

print(f'\nTraining samples: {train_size}')
print(f'Testing samples:  {test_size}')

✅ TrOCR processor loaded!


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded! Parameters: 333,921,792
Mapping image paths across all subdirectories...
⚠️ Filtered out 20 missing images.
✅ Final Dataset size: 403 samples.

Training samples: 322
Testing samples:  81


In [6]:
from torch.optim import AdamW

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

# Optimiser — AdamW with learning rate 5e-5 (standard for fine-tuning)
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('✅ Ready to train!')

Training batches per epoch: 81
✅ Ready to train!


In [7]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        # Move data to GPU
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass — model predicts text from images
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        # Backward pass — update model weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Print progress every 10 batches
        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\n✅ Training complete!')


Epoch 1/3
------------------------------
  Batch 0/81 | Loss: 19.2140
  Batch 10/81 | Loss: 0.7923
  Batch 20/81 | Loss: 1.4036
  Batch 30/81 | Loss: 0.3044
  Batch 40/81 | Loss: 0.9224
  Batch 50/81 | Loss: 1.2415
  Batch 60/81 | Loss: 0.1085
  Batch 70/81 | Loss: 0.2532
  Batch 80/81 | Loss: 0.0762
Epoch 1 complete | Average Loss: 0.8440

Epoch 2/3
------------------------------
  Batch 0/81 | Loss: 0.0795
  Batch 10/81 | Loss: 0.0742
  Batch 20/81 | Loss: 0.2213
  Batch 30/81 | Loss: 0.2538
  Batch 40/81 | Loss: 0.1726
  Batch 50/81 | Loss: 0.0980
  Batch 60/81 | Loss: 0.2813
  Batch 70/81 | Loss: 1.6008
  Batch 80/81 | Loss: 0.1941
Epoch 2 complete | Average Loss: 0.3620

Epoch 3/3
------------------------------
  Batch 0/81 | Loss: 0.1723
  Batch 10/81 | Loss: 0.0843
  Batch 20/81 | Loss: 0.1061
  Batch 30/81 | Loss: 0.0866
  Batch 40/81 | Loss: 0.2903
  Batch 50/81 | Loss: 0.0746
  Batch 60/81 | Loss: 0.2115
  Batch 70/81 | Loss: 0.0810
  Batch 80/81 | Loss: 0.0668
Epoch 3 compl

In [8]:
model.eval()

print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']

        # Generate predictions
        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )
        actual_text = processor.batch_decode(
            labels, skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual:    {actual}')
            match = '✓' if pred.strip() == actual.strip() else '✗'
            print(f'Match:     {match}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'🎯 Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')
print('\nRecord your accuracy and 3-5 failure examples for your report!')

=== Model Evaluation on Test Images ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1625: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Predicted: ؁�
Actual:    حرف
Match:     ✗

Predicted: 
Actual:    حرف
Match:     ✗

Predicted: 
Actual:    ا
Match:     ✗

Predicted: 
Actual:    حرف
Match:     ✗

Predicted: ؁�
Actual:    حرف
Match:     ✗

Predicted: 
Actual:    ب
Match:     ✗

Predicted: 
Actual:    حرف
Match:     ✗

Predicted: ح��
Actual:    حرف
Match:     ✗

Predicted: ؁�
Actual:    حرف
Match:     ✗

Predicted: ؁�
Actual:    حرف
Match:     ✗

Predicted: ؁
Actual:    حرف
Match:     ✗

Predicted: ��رر�
Actual:    حرف
Match:     ✗

Predicted: 
Actual:    ژ
Match:     ✗

Predicted: 
Actual:    حرف
Match:     ✗

Predicted: 
Actual:    ذ
Match:     ✗

Predicted: 
Actual:    ج
Match:     ✗

Predicted: 
Actual:    م
Match:     ✗

Predicted: 
Actual:    حرف
Match:     ✗

Predicted: 
Actual:    حرف
Match:     ✗

Predicted: 
Actual:    آ
Match:     ✗

Predicted: 
Actual:    حرف
Match:     ✗

Predicted: 
Actual:    ز
Match:     ✗

Predicted: ���                
Actual:    زیارت حملے کے خلاف دھرنا ختم
Match:     ✗

Predicted: �

In [10]:
from google.colab import drive

drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f'✅ Model saved to Google Drive: {save_path}')
print('You can load this model again without retraining!')
print('\nOpen https://drive.google.com and confirm the folder exists.')

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
You can load this model again without retraining!

Open https://drive.google.com and confirm the folder exists.


Training Metrics:

My model accuracy is 0.0%
Training loss went from 19.2140 to 0.0668 (Average loss went from 0.8440 in Epoch 1 to 0.2203 in Epoch 3)

## Week 4 Submission Summary Notes

- My model accuracy is 0.0%
- Training loss went from 19.2140 to 0.0668 (Average loss went from 0.8440 in Epoch 1 to 0.2203 in Epoch 3)